In [ ]:
# --- ensure repo-root cwd ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))
import warnings; warnings.filterwarnings('ignore')  # silence sklearn feature-name noise

# Forecast report — SoH actual vs custom model vs √t, to warranty (multi-page PDF)

One page per vehicle. Trains the condition-aware degradation model on all cohort vehicles, then
for each vehicle with enough history plots actual SoH, the custom median + 10–90% band, the √t
baseline, the 80% EOL line, and the warranty boundary. Output: `data/mahindra/soh/forecast_report.pdf`.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.dates import YearLocator, DateFormatter

WARRANTY_Y, EOL, MIN_MONTHS = 5.0, 80.0, 10
m = pd.read_parquet("data/mahindra/features/feature_table.parquet").sort_values(["vin", "month"])
STATE = ["soh", "age_months", "cum_ah", "cum_km", "odo_max"]
STRESS = ["ah_throughput", "cur_abs_mean", "cur_abs_p95", "cur_dis_mean", "cur_chg_mean",
          "soc_mean", "frac_soc_high", "frac_soc_low", "volt_mean", "volt_min", "volt_max",
          "temp_mean", "temp_max", "frac_drive", "km_month", "dte_mean"]
FEATS = STATE + STRESS

rows = []
for vin, g in m.groupby("vin"):
    g = g.sort_values("month").reset_index(drop=True)
    gap = g["month"].diff().shift(-1).dt.days / 30.4
    loss = ((g["soh"] - g["soh"].shift(-1)) / gap).clip(lower=0)
    r = g[FEATS].copy(); r["loss"] = loss.values; r["gap"] = gap.values
    rows.append(r)
tr = pd.concat(rows, ignore_index=True); tr = tr[(tr["gap"] <= 3) & tr["loss"].notna()]
models = {a: lgb.LGBMRegressor(objective="quantile", alpha=a, n_estimators=400, learning_rate=0.03,
                               num_leaves=15, min_child_samples=20, verbose=-1)
                .fit(tr[FEATS].to_numpy(), tr["loss"].to_numpy()) for a in (0.1, 0.5, 0.9)}
print(f"trained on {len(tr)} transitions from {m['vin'].nunique()} vehicles")

In [ ]:
def recent_stress(g, k=6):
    return g.sort_values("month").iloc[-k:][STRESS].median()

def simulate(g, horizon, eol=EOL):
    g = g.sort_values("month"); last = g.iloc[-1]; stress = recent_stress(g)
    st = {s: last[s] for s in STATE}; soh = {q: last["soh"] for q in (0.1, 0.5, 0.9)}; traj = []
    for _ in range(max(horizon, 1)):
        x = pd.DataFrame([{**{s: st[s] for s in STATE}, **stress.to_dict()}])[FEATS].to_numpy()
        for q in (0.1, 0.5, 0.9):
            soh[q] = soh[q] - max(models[q].predict(x)[0], 0)
        st.update(soh=soh[0.5], age_months=st["age_months"] + 1,
                  cum_ah=st["cum_ah"] + stress["ah_throughput"])
        traj.append((soh[0.9], soh[0.5], soh[0.1]))
    t = pd.DataFrame(traj, columns=["p90", "med", "p10"])
    cross = np.where(t["med"].to_numpy() <= eol)[0]
    return t, (int(cross[0] + 1) if len(cross) else None)

def sqrt_curve(g, ages):
    t = g["age_months"].to_numpy(); s = g["soh"].to_numpy(); mk = t > 0
    if mk.sum() < 2: return np.full(len(ages), s[-1])
    k = np.sum((100 - s[mk]) * np.sqrt(t[mk])) / np.sum(t[mk])
    return 100 - k * np.sqrt(ages)

# eligible vehicles, ranked by custom RUL (most-at-risk first)
elig = [(vin, g) for vin, g in m.groupby("vin") if len(g) >= MIN_MONTHS]
ranked = []
for vin, g in elig:
    last_age = g["age_months"].max(); horizon = int(WARRANTY_Y * 12 - last_age) + 1
    _, rul = simulate(g, horizon)
    ranked.append((rul if rul is not None else 9e9, vin, g))
ranked.sort(key=lambda z: z[0])
print(f"{len(ranked)} vehicles with >= {MIN_MONTHS} months -> one page each")

In [ ]:
out = "data/mahindra/soh/forecast_report.pdf"
with PdfPages(out) as pdf:
    # summary page
    fig = plt.figure(figsize=(11, 8)); fig.text(0.5, 0.93, "SoH Forecast Report — Mahindra cohort",
                                                ha="center", fontsize=16, weight="bold")
    summ = [f"{('EOL<warranty' if r < 9e9 else 'OK'):>13}  RUL={('%d mo'%r) if r<9e9 else '>warranty':>10}  "
            f"SoH_now={g.sort_values('month')['soh'].iloc[-1]:5.1f}%  {vin}" for r, vin, g in ranked]
    fig.text(0.07, 0.88, f"{len(ranked)} vehicles, sorted by predicted RUL (5-yr warranty, 80% EOL)\n\n" +
             "\n".join(summ[:48]), fontsize=7, family="monospace", va="top")
    pdf.savefig(fig); plt.close(fig)
    # one page per vehicle
    for r, vin, g in ranked:
        g = g.sort_values("month"); start = g["month"].min()
        t = g["age_months"].to_numpy(); s = g["soh"].to_numpy()
        horizon = int(WARRANTY_Y * 12 - t[-1]) + 1
        sim, rul = simulate(g, horizon); fut_age = t[-1] + np.arange(1, len(sim) + 1)
        d0 = start + pd.to_timedelta(t * 30.4375, unit="D")
        df_ = start + pd.to_timedelta(fut_age * 30.4375, unit="D")
        we = start + pd.DateOffset(years=int(WARRANTY_Y))
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.plot(d0, s, "o-", color="C0", ms=4, label="actual SoH (coulomb)")
        ax.plot(df_, sim["med"], "-", color="C2", lw=2, label="custom model (median)")
        ax.fill_between(df_, sim["p10"], sim["p90"], color="C2", alpha=0.18, label="custom 10–90%")
        ax.plot(df_, sqrt_curve(g, fut_age), "--", color="C3", lw=1.3, alpha=0.8, label="√t baseline")
        ax.axhline(EOL, ls=":", color="red"); ax.axvline(we, ls="-.", color="green", label=f"{WARRANTY_Y:.0f}-yr warranty")
        rul_txt = f"{rul} mo" if rul is not None else ">warranty"
        ax.set_title(f"{vin} ({g['vehicleModel'].dropna().iloc[0] if 'vehicleModel' in g and g['vehicleModel'].notna().any() else 'Treo'})"
                     f"  |  SoH now {s[-1]:.1f}%  |  custom RUL {rul_txt}")
        ax.set_ylim(70, 102); ax.set_ylabel("SoH (%)"); ax.set_xlabel("Date")
        ax.xaxis.set_major_locator(YearLocator()); ax.xaxis.set_major_formatter(DateFormatter("%Y"))
        ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=8)
        pdf.savefig(fig); plt.close(fig)
